# NB3 — Submission notebook (code competition)

**Settings:** GPU, Internet OFF. **Attach:** competition data (test set revealed at submit time) + datasets `rsna-knee-ckpt-v1` and `dinov2-timm-cache`.
Run-on-submit: the same code executes against the hidden test set.
Windowing/model MUST stay in sync with NB1/NB2 (source of truth: `rsna-knee/src/windows.py`, `src/model.py`).

In [ ]:
import os, glob
hub_dirs = (glob.glob('/kaggle/input/*/hf/hub') + glob.glob('/kaggle/input/*/*/hf/hub') + glob.glob('/kaggle/input/*/*/*/hf/hub'))
hub_dirs = [h for h in hub_dirs if glob.glob(h + '/models--timm--*')]
assert hub_dirs, f'timm cache not found under /kaggle/input: {sorted(os.listdir("/kaggle/input"))}'
hf_root = os.path.dirname(hub_dirs[0])
os.environ['HF_HOME'] = hf_root
os.environ['TORCH_HOME'] = os.path.join(os.path.dirname(hf_root), 'torch')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
print('HF_HOME =', hf_root)


In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import timm

comp_cands = (glob.glob('/kaggle/input/*/sample_submission.csv') + glob.glob('/kaggle/input/*/*/sample_submission.csv') + glob.glob('/kaggle/input/*/*/*/sample_submission.csv'))
assert comp_cands, f'competition data not attached; mounts={sorted(os.listdir("/kaggle/input"))}'
COMP = Path(os.path.dirname(comp_cands[0]))
WINDOWS_PER_STUDY = 42
SLICES_PER_WINDOW = 5
TARGET = 224
DEVICE = 'cuda'

sample = pd.read_csv(COMP / 'sample_submission.csv')
df_test = pd.read_csv(COMP / 'test.csv')
ts_csv = COMP / 'test_series.csv'
label_cols = [c for c in sample.columns if c != 'StudyInstanceUID']

if ts_csv.exists():
    df_series = pd.read_csv(ts_csv)
    series_by_study = {str(u): g['SeriesInstanceUID'].tolist() for u, g in df_series.groupby('StudyInstanceUID')}
else:
    series_by_study = {}
    base = COMP / 'test_series'
    for d in base.iterdir():
        if d.is_dir():
            series_by_study[str(d.name)] = [s.name for s in d.iterdir() if s.is_dir()]
print(f'{len(df_test)} test studies, series known for {len(series_by_study)}')

In [ ]:
# ---- windowing (verbatim from src/windows.py — keep in sync with NB1) ----
def normalize_percentile(arr, low=2.0, high=98.0):
    p_low = np.percentile(arr, low)
    p_high = np.percentile(arr, high)
    if p_high - p_low < 1e-6:
        return np.zeros_like(arr, dtype=np.float32)
    clipped = np.clip(arr, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low)).astype(np.float32)

def resize_volume(vol, target_size=224):
    d, h, w = vol.shape
    if h == target_size and w == target_size:
        return vol.astype(np.float32)
    resized = np.zeros((d, target_size, target_size), dtype=np.float32)
    for i in range(d):
        resized[i] = cv2.resize(vol[i], (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    return resized

def extract_series_windows(slices, n_windows, slices_per_window=5, target_size=224):
    num_slices = slices.shape[0]
    resized_slices = resize_volume(normalize_percentile(slices), target_size=target_size)
    if num_slices < slices_per_window:
        pad = np.repeat(resized_slices[-1:], slices_per_window - num_slices, axis=0)
        resized_slices = np.concatenate([resized_slices, pad], axis=0)
        num_slices = slices_per_window
    max_start = num_slices - slices_per_window
    if max_start == 0:
        start_indices = [0] * n_windows
    else:
        start_indices = np.linspace(0, max_start, n_windows, dtype=int).tolist()
    windows = np.zeros((n_windows, slices_per_window, target_size, target_size), dtype=np.float32)
    for idx, start in enumerate(start_indices):
        windows[idx] = resized_slices[start:start + slices_per_window]
    return windows

def sort_key(p):
    ds = pydicom.dcmread(str(p), stop_before_pixels=True)
    pos = getattr(ds, 'SliceLocation', None)
    if pos is not None:
        return (0, float(pos), 0.0)
    ipp = getattr(ds, 'ImagePositionPatient', None)
    if ipp is not None:
        return (1, 0.0, float(ipp[2]))
    return (2, 0.0, 0.0)

def study_windows(study_uid, series_uids, n_total):
    num_series = len(series_uids)
    counts = [0] * num_series
    for i in range(n_total):
        counts[i % num_series] += 1
    parts = []
    for series_uid, k in zip(series_uids, counts):
        sdir = COMP / 'test_series' / str(study_uid) / str(series_uid)
        slices_dir = sorted(sdir.glob('*.dcm'), key=sort_key)
        if not slices_dir:
            parts.append(np.zeros((k, SLICES_PER_WINDOW, TARGET, TARGET), dtype=np.float32))
            continue
        vol = np.stack([pydicom.dcmread(str(p)).pixel_array.astype(np.float32) for p in slices_dir])
        parts.append(extract_series_windows(vol, n_windows=k, slices_per_window=SLICES_PER_WINDOW, target_size=TARGET))
    return np.concatenate(parts, axis=0)

In [ ]:
# ---- model: timm backbone + MIL head from ckpt (verbatim src/model.py) ----
class GatedAttentionMIL(nn.Module):
    def __init__(self, in_features=384, hidden_dim=128, num_classes=12):
        super().__init__()
        self.attention_v = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Tanh())
        self.attention_u = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Sigmoid())
        self.attention_weights = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Linear(in_features, num_classes)
    def forward(self, h):
        v = self.attention_v(h); u = self.attention_u(h)
        a = self.attention_weights(v * u)
        w = torch.softmax(a, dim=1)
        m = torch.sum(w * h, dim=1)
        return self.classifier(m)

# resolve ckpt file robustly (dataset path may nest)
pats = ['/kaggle/input/rsna-knee-ckpt*/**/*.pt', '/kaggle/input/*/rsna-knee-ckpt*/**/*.pt', '/kaggle/input/*/*/rsna-knee-ckpt*/**/*.pt', '/kaggle/input/*/*/*/rsna-knee-ckpt*/**/*.pt']
ck_files = []
for pat in pats:
    ck_files += glob.glob(pat, recursive=True)
assert ck_files, f'checkpoint not found; mounts={sorted(os.listdir("/kaggle/input"))}'
ck_path = ck_files[0]
ck = torch.load(ck_path, map_location='cpu', weights_only=False)
hidden = int(ck.get('hidden', 128))
num_classes = ck.get('num_classes', len(label_cols))
label_cols_ck = ck.get('label_cols')
print('ckpt label_cols match:', label_cols_ck is None or list(label_cols_ck) == label_cols)

backbone = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, in_chans=5, img_size=224, num_classes=0).to(DEVICE).eval()
PREFIX = int(getattr(backbone, 'num_prefix_tokens', 1))
sd = None
def make_head(sd):
    h = GatedAttentionMIL(384, hidden, num_classes).to(DEVICE).eval()
    mil_sd = {k[len('mil.'):] if k.startswith('mil.') else k: v for k, v in sd.items() if k.startswith('mil.')}
    h.load_state_dict(mil_sd or sd)
    return h

sds = ck.get('state_dicts')
heads = [make_head(s) for s in sds] if sds else [make_head(ck.get('state_dict') or ck.get('model_state'))]
print('ckpt val AUC:', ck.get('val_auc'), '| heads loaded:', len(heads))

In [ ]:
@torch.no_grad()
def predict_study(study_uid):
    series_uids = series_by_study.get(str(study_uid), [])
    if series_uids:
        win = study_windows(study_uid, series_uids, WINDOWS_PER_STUDY)
    else:
        win = np.zeros((WINDOWS_PER_STUDY, SLICES_PER_WINDOW, TARGET, TARGET), dtype=np.float32)
    embs = []
    for i in range(0, len(win), 64):
        x = torch.from_numpy(win[i:i+64]).to(DEVICE)
        feats = backbone.forward_features(x)
        embs.append(feats[:, PREFIX:].mean(dim=1))
    emb = torch.cat(embs, dim=0).unsqueeze(0)          # [1, W, 384]
    logits = torch.stack([h(emb) for h in heads], 0).mean(0)
    return torch.sigmoid(logits)[0].cpu().numpy()

rows = []
for study_uid in sample['StudyInstanceUID'].astype(str):
    probs = predict_study(study_uid)
    rows.append([study_uid] + [float(p) for p in probs])

submission = pd.DataFrame(rows, columns=sample.columns)
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.shape)
submission.head()